# CSE 151B — SFT training (Numina bf16 LoRA, A100)

bf16 LoRA fine-tune `Qwen/Qwen3-4B-Thinking-2507` on a named corpus (set `RUN_NAME` + paths in §2). Current config: **`openmath_sft007_5k`** (sft-007) — OpenMathReasoning 1k sequences-only, 12k–28k char traces. Alternates: `openmath_geo_1k` (sft-005), `openmath_seq_1k` (sft-006), `openmath_1k` (sft-003).

**Workflow (Colab A100):**
1. Run `%pip` → restart runtime.
2. Mount Drive; copy corpus + eval files (§3).
3. Set `SMOKE_MODE = True` → smoke train (≤100 steps, ≤500 rows) → inspect loss + §9 samples.
4. Set `SMOKE_MODE = False` → full 1-epoch run; checkpoints every 500 steps to Drive.
5. **§8** runs holdout + dev-slice eval on `final_adapter` (vLLM + Judger; no separate `sft_eval.ipynb` required).
6. Optional: re-run §8 only after pointing `OUTPUT_DIR` at an existing adapter.

Plan: [`docs/sft/pipeline.md`](../docs/sft/pipeline.md). Previous runs: openmath_1k (sft-003, regressive), openr1_1k (sft-002a, flat).



## 1. Environment

**Colab:** run `%pip`, restart, continue. **Local:** install `torch`, `transformers`, `peft`, `trl`, `accelerate`, `datasets` in your venv. (No `bitsandbytes` — bf16 LoRA on A100 only.)

In [7]:
# # Colab: skip when working locally.
# %pip install -q sympy numpy tqdm datasets accelerate
# %pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu129 --upgrade
# %pip install -q transformers peft trl "torchao>=0.16.0"

## 2. Configuration

Toggle `SMOKE_MODE` for the first Colab session. `RUN_NAME` labels Drive checkpoints and `sft_eval` runs.

In [2]:
import json
import os
import shutil
from pathlib import Path


def _repo_root() -> Path:
    here = Path.cwd().resolve()
    if (here / "data").is_dir():
        return here
    if here.name == "notebooks" and (here.parent / "data").is_dir():
        return here.parent
    up = here.parent
    if (up / "data").is_dir():
        return up
    return here


REPO_ROOT = _repo_root()

# --- run switches (sft-007) ---
SMOKE_MODE = False
RUN_NAME = "openmath_sft007_5k"

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
CORPUS_PATH = REPO_ROOT / "data/sft_corpus_sft007_openmath_5k.jsonl"
MANIFEST_PATH = REPO_ROOT / "data/sft_corpus_sft007_openmath_5k_manifest.json"
EVAL_HOLDOUT_PATH = REPO_ROOT / "data/eval/holdout_20p.jsonl"

# Drive layout mirrors repo (see pipeline.md)
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/CSE151B")
OUTPUT_DIR = DRIVE_PROJECT_ROOT / "checkpoints" / RUN_NAME

# --- hyperparameters (sft-007 gentle LoRA) ---
LORA_R = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
LORA_TARGETS = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]
LEARNING_RATE = 5e-6
WARMUP_RATIO = 0.03
PER_DEVICE_BATCH = 1
GRAD_ACCUM = 16
SAVE_STEPS = 50
LOGGING_STEPS = 10

# --- §8 inline eval ---
RUN_EVAL_AFTER_TRAIN = True
EVAL_MAX_TOKENS = 16384
GATE_HOLDOUT_OVERALL = 0.6444
GATE_HOLDOUT_MCQ = 0.77
PUB002_GEOMETRY_ACC = 53.38
PUB002_PROB_STATS_ACC = 49.8
EVAL_SLICES = [
    ("holdout", REPO_ROOT / "data/eval/holdout_20p.jsonl"),
    ("geometry", REPO_ROOT / "data/eval/geometry_dev.jsonl"),
    ("prob_stats", REPO_ROOT / "data/eval/prob_stats_dev.jsonl"),
]

if SMOKE_MODE:
    MAX_SAMPLES = 500
    MAX_STEPS = 100
    MAX_SEQ_LENGTH = 4096
    NUM_TRAIN_EPOCHS = 1
else:
    MAX_SAMPLES = None
    MAX_STEPS = -1
    MAX_SEQ_LENGTH = 8192
    NUM_TRAIN_EPOCHS = 1

SEED = 42
GPU_ID = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print(f"REPO_ROOT={REPO_ROOT}")
print(f"SMOKE_MODE={SMOKE_MODE} RUN_NAME={RUN_NAME}")
print(f"OUTPUT_DIR={OUTPUT_DIR}")
print(f"RUN_EVAL_AFTER_TRAIN={RUN_EVAL_AFTER_TRAIN}")



REPO_ROOT=/content
SMOKE_MODE=False RUN_NAME=openmath_sft007_5k
OUTPUT_DIR=/content/drive/MyDrive/CSE151B/checkpoints/openmath_sft007_5k
RUN_EVAL_AFTER_TRAIN=True


## 3. Colab: Drive + corpus copy

In [3]:
try:
    from google.colab import drive
except ImportError:
    print("Skip: not Colab — use repo paths.")
else:
    drive.mount("/content/drive")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for rel in [
        "data/sft_corpus_sft007_openmath_5k.jsonl",
        "data/sft_corpus_sft007_openmath_5k_manifest.json",
        "data/eval/holdout_20p.jsonl",
        "data/eval/geometry_dev.jsonl",
        "data/eval/prob_stats_dev.jsonl",
        "data/eval/watch_q4_long.jsonl",
        "data/eval/watch_multi_blank_ge3.jsonl",
        "data/eval/watch_manifest.json",
    ]:
        src = DRIVE_PROJECT_ROOT / rel
        dst = REPO_ROOT / rel
        if src.is_file():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
            print(f"Copied {rel}")
        elif not dst.is_file():
            print(f"Warning: missing {src} and {dst}")

if not CORPUS_PATH.is_file():
    raise FileNotFoundError(
        f"Missing {CORPUS_PATH}. Run sft_data_prep §10.5 or scripts/build_sft_corpus_sft007.py."
    )
if MANIFEST_PATH.is_file():
    _manifest = json.loads(MANIFEST_PATH.read_text())
    print(
        f"Corpus manifest: {_manifest.get('final_row_count')} rows, "
        f"slices={_manifest.get('slice_counts')}"
    )



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copied data/sft_corpus_sft007_openmath_5k.jsonl
Copied data/sft_corpus_sft007_openmath_5k_manifest.json
Copied data/eval/holdout_20p.jsonl
Copied data/eval/geometry_dev.jsonl
Copied data/eval/watch_q4_long.jsonl
Copied data/eval/watch_multi_blank_ge3.jsonl
Copied data/eval/watch_manifest.json
Corpus manifest: 5000 rows, slices={'trigonometry': 1000, 'geometry': 1200, 'probability/stats': 1300, 'integration': 127, 'linear algebra': 29, 'polynomials/algebra': 402, 'logs/exponents': 24, 'other': 563, 'number theory': 201, 'arithmetic/word problems': 51, 'limits': 62, 'derivatives': 18, 'complex analysis': 23}


## 4. Prompt helpers (match pub-002 / `full_public.ipynb`)

Uses `build_prompt_multi_blank` from `scripts/sft_prompt.py` so multi-blank corpus rows (2+ `[ANS]`) get the same system/user wrapper as inference.

In [4]:
from typing import Any, Optional


_MATH_BASELINE = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

_MCQ_BASELINE = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)

_MATH_MULTI_BLANK = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "For problems with multiple [ANS] placeholders, put each sub-answer in its own "
    "\\boxed{}, separated by commas, in the order the blanks appear "
    "(e.g. \\boxed{3}, \\boxed{7}). Do not use labels like 'Answer 1:' between boxes. "
    "For single-answer problems, use one \\boxed{}."
)

def n_ans_placeholders(question: str) -> int:
    return question.count("[ANS]")

def build_prompt_baseline(question: str, options: Optional[list]) -> tuple[str, str]:
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return _MCQ_BASELINE, f"{question}\n\nOptions:\n{opts_text}"
    return _MATH_BASELINE, question

def build_prompt_multi_blank(question: str, options: Optional[list]) -> tuple[str, str]:
    """Inference-time multi-blank path (pub-002 / full_public.ipynb)."""
    if options:
        return build_prompt_baseline(question, options)
    n_blanks = n_ans_placeholders(question)
    if n_blanks <= 1:
        return build_prompt_baseline(question, options)
    example = ", ".join("\\boxed{...}" for _ in range(n_blanks))
    user = (
        f"{question}\n\n"
        f"The problem has {n_blanks} [ANS] blanks. "
        f"After your reasoning, give {n_blanks} comma-separated \\boxed{{}} values "
        f"in order: {example}"
    )
    return _MATH_MULTI_BLANK, user

# pub-002 / full_public inference path; required for sft_corpus_v2 multi-blank rows
build_prompt = build_prompt_multi_blank


def row_to_messages(row: dict[str, Any]) -> list[dict[str, str]]:
    system, user = build_prompt(row["question"], row.get("options"))
    return [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
        {"role": "assistant", "content": row["response"]},
    ]


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

## 5. Build Hugging Face dataset

In [5]:
import torch
import random
from datasets import Dataset
from transformers import AutoTokenizer

random.seed(SEED)
corpus_rows = read_jsonl(CORPUS_PATH)
random.shuffle(corpus_rows)
if MAX_SAMPLES is not None:
    corpus_rows = corpus_rows[:MAX_SAMPLES]

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# TRL assistant_only_loss needs {% generation %} markers; Thinking-2507 is not an exact
# match for get_training_chat_template(), so use the Qwen3 thinking training template.
INFERENCE_CHAT_TEMPLATE = tokenizer.chat_template
from trl.chat_template_utils import qwen3_training_chat_template

tokenizer.chat_template = qwen3_training_chat_template

records = [{"messages": row_to_messages(r)} for r in corpus_rows]
train_ds = Dataset.from_list(records)

# Drop rows that exceed max length after chat template
def _token_len(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"n_tokens": len(tokenizer.encode(text, add_special_tokens=False))}

train_ds = train_ds.map(_token_len, desc="token lengths")
before = len(train_ds)
train_ds = train_ds.filter(lambda x: x["n_tokens"] <= MAX_SEQ_LENGTH)
train_ds = train_ds.remove_columns(["n_tokens"])
print(f"Train rows: {len(train_ds)} (dropped {before - len(train_ds)} over {MAX_SEQ_LENGTH} tokens)")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


token lengths:   0%|          | 0/5000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5000 [00:00<?, ? examples/s]

Train rows: 5000 (dropped 0 over 8192 tokens)


## 6. Load bf16 base + LoRA

In [6]:
from peft import LoraConfig
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
model.gradient_checkpointing_enable()

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS,
    bias="none",
    task_type="CAUSAL_LM",
)
print("Base model (bf16) + LoRA config ready.")

print(model.config._attn_implementation)


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Base model (bf16) + LoRA config ready.
sdpa


## 7. Train (TRL SFTTrainer, assistant-only loss)

Resumes from the latest `checkpoint-*` in `OUTPUT_DIR` when present.

In [12]:
from datetime import datetime, timezone

from trl import SFTConfig, SFTTrainer


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

_checkpoints = sorted(OUTPUT_DIR.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
resume_ckpt = str(_checkpoints[-1]) if _checkpoints else None
if resume_ckpt:
    print(f"Resuming from {resume_ckpt}")

_effective_batch = PER_DEVICE_BATCH * GRAD_ACCUM
_steps_per_epoch = max(1, len(train_ds) // _effective_batch)
_total_steps = _steps_per_epoch * NUM_TRAIN_EPOCHS
if MAX_STEPS > 0:
    _total_steps = MAX_STEPS
_warmup_steps = max(1, int(_total_steps * WARMUP_RATIO))
print(f"Training steps: {_total_steps} (warmup_steps={_warmup_steps})")

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=_warmup_steps,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    max_steps=MAX_STEPS if MAX_STEPS > 0 else -1,
    max_length=MAX_SEQ_LENGTH,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    bf16=True,
    optim="adamw_torch_fused",
    gradient_checkpointing=True,
    report_to="none",
    seed=SEED,
    dataset_text_field=None,
    assistant_only_loss=True,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    processing_class=tokenizer,
    peft_config=peft_config,
)

train_result = trainer.train(resume_from_checkpoint=resume_ckpt)
print(train_result)

final_dir = OUTPUT_DIR / "final_adapter"
trainer.save_model(str(final_dir))
tokenizer.save_pretrained(str(final_dir))

run_meta = {
    "run_name": RUN_NAME,
    "smoke_mode": SMOKE_MODE,
    "model_id": MODEL_ID,
    "corpus_path": str(CORPUS_PATH),
    "train_rows": len(train_ds),
    "max_seq_length": MAX_SEQ_LENGTH,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "learning_rate": LEARNING_RATE,
    "warmup_steps": _warmup_steps,
    "warmup_ratio": WARMUP_RATIO,
    "finished_at": datetime.now(timezone.utc).isoformat(),
    "final_adapter": str(final_dir),
}
(OUTPUT_DIR / "run_meta.json").write_text(json.dumps(run_meta, indent=2) + "\n")
print(f"Saved adapter → {final_dir}")

Training steps: 312 (warmup_steps=9)


Tokenizing train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,0.652788
20,0.637047
30,0.644014
40,0.628789
50,0.581967
60,0.551135
70,0.537726
80,0.526237
90,0.515285
100,0.496592


TrainOutput(global_step=313, training_loss=0.5170185882062577, metrics={'train_runtime': 9332.4872, 'train_samples_per_second': 0.536, 'train_steps_per_second': 0.034, 'total_flos': 6.382985386951588e+17, 'train_loss': 0.5170185882062577, 'entropy': 0.46923600435256957, 'num_tokens': 28755511.0, 'mean_token_accuracy': 0.8301689758896827, 'epoch': 1.0})
Saved adapter → /content/drive/MyDrive/CSE151B/checkpoints/openmath_sft007_5k/final_adapter


## 8. Post-train eval (vLLM + Judger)

Runs immediately after §7 when `RUN_EVAL_AFTER_TRAIN=True`. Same stack as `sft_eval.ipynb` / pub-002 (`multi_blank`, 16k). Gate: holdout overall ≥ 64.44%, MCQ ≥ 77%. Re-run this section alone to test an existing `final_adapter`.


In [5]:
import gc
import glob
import re
import sys
from typing import Optional

if not RUN_EVAL_AFTER_TRAIN:
    print("Skipping §8 (RUN_EVAL_AFTER_TRAIN=False)")
else:
    final_dir = OUTPUT_DIR / "final_adapter"
    if not final_dir.is_dir():
        raise FileNotFoundError(f"Missing adapter: {final_dir}")

    for _name in ("trainer", "model", "train_ds"):
        if _name in globals():
            del globals()[_name]
    gc.collect()
    try:
        import torch
        torch.cuda.empty_cache()
    except Exception:
        pass

    os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

    import contextlib
    import io

    @contextlib.contextmanager
    def _jupyter_stdout_for_vllm():
        try:
            sys.stdout.fileno()
        except (io.UnsupportedOperation, AttributeError, OSError):
            saved_out, saved_err = sys.stdout, sys.stderr
            dup1, dup2 = os.dup(1), os.dup(2)
            try:
                sys.stdout = os.fdopen(dup1, "w", buffering=1)
                sys.stderr = os.fdopen(dup2, "w", buffering=1)
                yield
            finally:
                sys.stdout.close()
                sys.stderr.close()
                sys.stdout, sys.stderr = saved_out, saved_err
        else:
            yield

    def _prepend_nvidia_libs_to_ld_path() -> None:
        import site
        roots = list(site.getsitepackages())
        user_site = getattr(site, "getusersitepackages", lambda: None)()
        if user_site:
            roots.append(user_site)
        libdirs: list[str] = []
        seen: set[str] = set()
        for root in roots:
            for d in glob.glob(os.path.join(root, "nvidia", "*", "lib")):
                if os.path.isdir(d) and d not in seen:
                    seen.add(d)
                    libdirs.append(d)
        if libdirs:
            sep = os.pathsep
            os.environ["LD_LIBRARY_PATH"] = sep.join(
                libdirs + [os.environ.get("LD_LIBRARY_PATH", "")]
            ).strip(sep)

    _prepend_nvidia_libs_to_ld_path()

    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams

    eval_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    eval_tokenizer.pad_token = eval_tokenizer.eos_token

    _llm_kwargs = dict(
        model=MODEL_ID,
        dtype="bfloat16",
        enable_prefix_caching=True,
        gpu_memory_utilization=0.92,
        max_model_len=17500,
        trust_remote_code=True,
        max_num_seqs=128,
        max_num_batched_tokens=32768,
        enable_chunked_prefill=True,
        enable_lora=True,
        max_loras=1,
        max_lora_rank=64,
    )
    with _jupyter_stdout_for_vllm():
        llm = LLM(**_llm_kwargs)

    from vllm.lora.request import LoRARequest
    lora_request = LoRARequest("sft_adapter", 1, str(final_dir))
    print(f"LoRA adapter: {final_dir}")

    sampling_params = SamplingParams(
        max_tokens=EVAL_MAX_TOKENS,
        temperature=0.6,
        top_p=0.95,
        top_k=20,
        seed=SEED,
    )

    sys.path.insert(0, str(REPO_ROOT))
    from judger import Judger
    judger = Judger(strict_extract=False)

    def _read_eval_jsonl(path: Path) -> list[dict]:
        with open(path) as f:
            return [json.loads(line) for line in f if line.strip()]

    def _mcq_has_boxed_letter(response: str, n_options: int) -> bool:
        last = chr(64 + n_options)
        return bool(re.search(rf"\\boxed\\{{[A-{last}]\\}}", response, re.IGNORECASE))

    _watch_manifest_path = REPO_ROOT / "data/eval/watch_manifest.json"
    with open(_watch_manifest_path) as f:
        _watch_manifest = json.load(f)

    def _watch_ids(name: str) -> set[int]:
        return set(_watch_manifest["watch"][name]["ids"])

    _eval_root = DRIVE_PROJECT_ROOT / "results" / "sft_eval" / RUN_NAME
    _eval_root.mkdir(parents=True, exist_ok=True)

    _slice_summaries: list[dict] = []
    _holdout_overall = _holdout_mcq = None

    for slice_name, data_path in EVAL_SLICES:
        if not data_path.is_file():
            raise FileNotFoundError(data_path)
        data = _read_eval_jsonl(data_path)
        ckpt = _eval_root / f"eval_{slice_name}_0.responses.jsonl"
        completed: dict[int, str] = {}
        if ckpt.is_file():
            with open(ckpt) as f:
                for line in f:
                    r = json.loads(line)
                    completed[r["id"]] = r["response"]
        remaining = [d for d in data if d["id"] not in completed]
        print(f"\n=== eval slice={slice_name} n={len(data)} remaining={len(remaining)} ===")

        for item in remaining:
            system, user = build_prompt(item["question"], item.get("options"))
            prompt_text = eval_tokenizer.apply_chat_template(
                [{"role": "system", "content": system}, {"role": "user", "content": user}],
                tokenize=False,
                add_generation_prompt=True,
            )
            with _jupyter_stdout_for_vllm():
                outs = llm.generate(
                    prompts=[prompt_text],
                    sampling_params=sampling_params,
                    lora_request=lora_request,
                )
            response = outs[0].outputs[0].text.strip()
            completed[item["id"]] = response
            with open(ckpt, "a") as f:
                f.write(json.dumps({"id": item["id"], "response": response}) + "\n")

        results = []
        for item in data:
            resp = completed[item["id"]]
            gold = item["answer"]
            is_mcq = bool(item.get("options"))
            if is_mcq:
                correct = _extract_letter(resp) == str(gold).strip().upper()
            else:
                gold_list = gold if isinstance(gold, list) else [gold]
                try:
                    correct = judger.auto_judge(
                        pred=resp,
                        gold=gold_list,
                        options=[[]] * len(gold_list),
                    )
                except Exception:
                    correct = False
            results.append({
                "id": item["id"],
                "is_mcq": is_mcq,
                "gold": gold,
                "response": resp,
                "correct": correct,
            })

        mcq_res = [r for r in results if r["is_mcq"]]
        free_res = [r for r in results if not r["is_mcq"]]

        def _acc(subset):
            return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

        overall = _acc(results)
        mcq_acc = _acc(mcq_res)
        mean_len = sum(len(r["response"]) for r in results) / len(results)

        print(f"  MCQ       : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({mcq_acc:.2f}%)")
        print(f"  Free-form : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({_acc(free_res):.2f}%)")
        print(f"  Overall   : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({overall:.2f}%)")
        print(f"  Mean len  : {mean_len:.0f} chars")

        if slice_name == "holdout":
            _holdout_overall, _holdout_mcq = overall, mcq_acc
            for wname, label in (
                ("q4_long", "Q4 long"),
                ("multi_blank_ge3", "Multi-blank ≥3"),
            ):
                ids = _watch_ids(wname)
                sub = [r for r in results if r["id"] in ids]
                wacc = _acc(sub)
                print(f"  {label:16s}: {sum(r['correct'] for r in sub):4d} / {len(sub):4d}  ({wacc:.2f}%)")
        elif slice_name == "geometry":
            print(f"  (pub-002 ref ~{PUB002_GEOMETRY_ACC:.2f}%)")
        elif slice_name == "prob_stats":
            print(f"  (pub-002 ref ~{PUB002_PROB_STATS_ACC:.2f}% full-public topic slice)")

        out_jsonl = _eval_root / f"eval_{slice_name}_0.jsonl"
        with open(out_jsonl, "w") as f:
            for r in results:
                f.write(json.dumps({
                    "id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                    "response": r["response"], "correct": r["correct"],
                }) + "\n")
        metrics = {
            "run_name": RUN_NAME,
            "eval_slice": slice_name,
            "overall_acc": overall,
            "mcq_acc": mcq_acc,
            "free_acc": _acc(free_res),
            "mean_response_len": mean_len,
            "n": len(results),
        }
        metrics_path = _eval_root / f"eval_{slice_name}_0.json"
        metrics_path.write_text(json.dumps(metrics, indent=2) + "\n")
        _slice_summaries.append(metrics)
        print(f"  Saved → {out_jsonl}")

    print("\n" + "=" * 50)
    print("GATE (holdout)")
    if _holdout_overall is None:
        print("  (no holdout run)")
    else:
        ok_o = _holdout_overall >= GATE_HOLDOUT_OVERALL * 100
        ok_m = _holdout_mcq >= GATE_HOLDOUT_MCQ * 100
        print(f"  Overall {_holdout_overall:.2f}% vs {GATE_HOLDOUT_OVERALL*100:.2f}% → {'PASS' if ok_o else 'FAIL'}")
        print(f"  MCQ     {_holdout_mcq:.2f}% vs {GATE_HOLDOUT_MCQ*100:.2f}% → {'PASS' if ok_m else 'FAIL'}")
    print("=" * 50)



ModuleNotFoundError: No module named 'vllm'

## 9. Loss curve (smoke / debug)

In [ ]:
import matplotlib.pyplot as plt

log_hist = trainer.state.log_history
steps, losses = [], []
for entry in log_hist:
    if "loss" in entry and "step" in entry:
        steps.append(entry["step"])
        losses.append(entry["loss"])
if losses:
    plt.figure(figsize=(8, 3))
    plt.plot(steps, losses, marker=".")
    plt.xlabel("step")
    plt.ylabel("loss")
    plt.title(f"{RUN_NAME} train loss")
    plt.grid(True, alpha=0.3)
    plt.show()
    print(f"First loss={losses[0]:.4f}  last loss={losses[-1]:.4f}")
else:
    print("No loss entries in log_history yet.")

## 10. Smoke: generate 10 dev samples

Quick qualitative check before full training. Expect `<think>` blocks and `\boxed{...}` finals. If outputs are answer-only or very short, stop and inspect schema / masking.

In [ ]:
from peft import PeftModel

N_SMOKE_SAMPLES = 1
MAX_NEW_TOKENS = 2048

if not EVAL_HOLDOUT_PATH.is_file():
    print(f"Skip smoke gen: {EVAL_HOLDOUT_PATH} missing")
else:
    dev_rows = read_jsonl(EVAL_HOLDOUT_PATH)[:N_SMOKE_SAMPLES]
    gen_model = PeftModel.from_pretrained(model, str(final_dir))
    gen_model.eval()

    for i, item in enumerate(dev_rows):
        system, user = build_prompt(item["question"], item.get("options"))
        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ]
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            chat_template=INFERENCE_CHAT_TEMPLATE,
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(gen_model.device)
        with torch.no_grad():
            out = gen_model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=0.6,
                top_p=0.95,
                pad_token_id=tokenizer.pad_token_id,
            )
        text = tokenizer.decode(out[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True)
        has_think = "<think>" in text
        has_boxed = "\\boxed" in text
        print(f"\n--- dev id={item['id']} think={has_think} boxed={has_boxed} len={len(text)} ---")
        print(text)

## 11. Next steps

1. **Smoke pass:** `SMOKE_MODE=True` — loss should trend down; §10 samples should keep thinking + `\boxed{}`.
2. **Full run:** `SMOKE_MODE=False`, A100 40GB+, `MAX_SEQ_LENGTH=16384`.
3. **Eval:** §8 runs automatically when `RUN_EVAL_AFTER_TRAIN=True`; or re-run §8 only on an existing adapter.
4. **Submit:** if §8 gates pass and a dev slice beats pub-002, point `submission.ipynb` at the same adapter.


In [ ]:
try:
    from google.colab import drive, runtime
    drive.flush_and_unmount()
    print("Drive flushed and unmounted.")
    runtime.unassign()
except ImportError:
    print("Not running in Colab — skipping runtime termination.")